# Tools

_Setup_

In [56]:
# Setup
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

claude_api_key = os.getenv("CLAUDE_API_KEY")

# client = Anthropic(base_url="http://localhost:11434", api_key="ollama")
client = Anthropic(api_key=claude_api_key)

# model = "gemma4:e4b-mlx"
model = "claude-sonnet-4-5"
max_tokens = 1000

_Utility Functions_

In [57]:
# Utility function
def add_user_message(messages, message):
  messages.append({"role": "user", "content": message})

from anthropic.types import Message
def add_assistant_message(messages, message):
  messages.append({"role": "assistant", "content": message.content if isinstance(message, Message) else [message] })

def chat(messages, **kargs):
  params = {
    "model": model,
    "max_tokens": max_tokens,
    "messages": messages,
    **kargs
  }

  response = client.messages.create(**params)
  return response

In [58]:
# print json
import json
def print_json(_a):
  print(json.dumps(_a, indent=2))

In [59]:
# Tools - get_current_date_time
from datetime import datetime
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
  if not date_format:
    raise ValueError("date_format cannot be empty")
  return datetime.now().strftime(date_format)

# schema
# In the end schema is a JSON, but to avoid spelling or type mistakes, 
# use the following type from anthropic library
from anthropic.types import ToolParam 
get_current_datetime_schema = ToolParam({
  "name": "get_current_datetime",
  "description": "Returns the current date and time formatted according to the specified format",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "required": []
  }
})

In [60]:
# Tools - add_duration_to_datetime

from datetime import datetime, timedelta
 
def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
  ):
    date = datetime.strptime(datetime_str, input_format)
 
    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")
 
    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")

#  Schema
add_duration_to_datetime_schema = {
  "name": "add_duration_to_datetime",
  "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
  "input_schema": {
    "type": "object",
    "properties": {
      "datetime_str": {
        "type": "string",
        "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
      },
      "duration": {
        "type": "number",
        "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
      },
      "unit": {
        "type": "string",
        "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
      },
      "input_format": {
        "type": "string",
        "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
      },
    },
    "required": ["datetime_str"],
  },
}

In [61]:
# Tools - set_reminder
def set_reminder(content, timestamp):
  print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")

set_reminder_schema = {
  "name": "set_reminder",
  "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
  "input_schema": {
    "type": "object",
    "properties": {
      "content": {
        "type": "string",
        "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
      },
      "timestamp": {
        "type": "string",
        "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
      },
    },
    "required": ["content", "timestamp"],
  },
}

In [77]:
# Tools - set_reminder
def get_weather():
  return "beautiful day"

get_weather_schema = {
  "name": "get_weather",
  "description": "Get the current weather",
  "input_schema": {
    "type": "object",
    "properties": {},
    "required": []
  },
}

In [80]:
# Tool Runner

tools = {
  'get_current_datetime': get_current_datetime,
  'add_duration_to_datetime': add_duration_to_datetime,
  'set_reminder': set_reminder,
  'get_weather': get_weather
}

def run_tool(tool_name, inputs):
  tool = tools.get(tool_name)
  if tool is not None:
    return tool(**inputs)
  else: raise Exception("invalid tool name")

In [ ]:
# Sample - how to use the tools
messages = []
add_user_message(messages, "set a reminder 3 days from today to visit the dental clinic")

resp = chat(messages, tools=[get_current_datetime_schema, add_duration_to_datetime_schema, set_reminder_schema])
print_json(resp.to_dict())

In [64]:
# Model Response
sample_response = {
  "id": "msg_011CekH6E2to2ifH4u5ghuPn",
  "content": [
    {
      "text": "I'll help you set a reminder for 3 days from today to visit the dental clinic. Let me first get the current date and then calculate the date 3 days from now.",
      "type": "text"
    },
    {
      "id": "toolu_01E4hw5U3di9huUpfmVzXzku",
      "caller": {
        "type": "direct"
      },
      "input": {},
      "name": "get_current_datetime",
      "type": "tool_use"
    }
  ],
  "model": "claude-sonnet-4-5-20250929",
  "role": "assistant",
  "stop_details": None,
  "stop_reason": "tool_use",
  "stop_sequence": None,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "not_available",
    "input_tokens": 1367,
    "output_tokens": 78,
    "service_tier": "standard"
  }
}


> When tools are provided, the chat function returns 1 or more tool_use blocks with type="tool_use".

> And the response generation stopped, we need to see the stop reason: if it's "tool_use" means the model stopped because it needs some more information from the tools. And we need to call the requested tools and respond back to the model with the tool call results

In [68]:
def text_from_message(response):
  for block in response.content:
    if (block.type == "text"): 
      return block.text
  return ""

def run_tools(response): 
  tool_result_blocks = []

  tool_requests = [block for block in response.content if block.type == 'tool_use']

  for tool_request in tool_requests:
    try:
      tool_output = run_tool(tool_request.name, tool_request.input)
      tool_result_block = {
        "type": "tool_result",
        "tool_use_id": tool_request.id,
        "content": tool_output,
        "is_error": False,
      }
      tool_result_blocks.append(tool_result_block)

    except Exception as e:
      tool_result_block = {
        "type": "tool_result",
        "tool_use_id": tool_request.id,
        "content": f"Error: {e}",
        "is_error": True,
      }
      tool_result_blocks.append(tool_result_block)

  return tool_result_blocks

In [78]:
def run_conversation(messages):
  while True:
    resp = chat(messages, tools=[get_current_datetime_schema, add_duration_to_datetime_schema, set_reminder_schema, get_weather_schema])
    add_assistant_message(messages, resp)
    print(text_from_message(resp))

    if resp.stop_reason != "tool_use":
        break

    tool_results = run_tools(resp)
    add_user_message(messages, tool_results)
 
  return messages


### The following one calls 1 tool at a time, because 1 tool call depends on another

In [ ]:
messages = []
add_user_message(messages, "set a reminder 3 days from today to visit the dental clinic")
run_conversation(messages)

### The following one calls multiple tools at the same time, because the tools are independent

In [81]:
messages = []
add_user_message(messages, "What is the current weather and current time?")
run_conversation(messages)
messages

I'll get the current weather and time for you.
The current time is **7:27:21 PM** (September 5, 2026) and the weather is a **beautiful day**!


[{'role': 'user', 'content': 'What is the current weather and current time?'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll get the current weather and time for you.", type='text'),
   ToolUseBlock(id='toolu_01GV9SZHGK4o7citBttUdiRF', caller=DirectCaller(type='direct'), input={}, name='get_weather', type='tool_use'),
   ToolUseBlock(id='toolu_01TgJxyvfRV6jFoSMh1A5HZy', caller=DirectCaller(type='direct'), input={}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01GV9SZHGK4o7citBttUdiRF',
    'content': 'beautiful day',
    'is_error': False},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_01TgJxyvfRV6jFoSMh1A5HZy',
    'content': '2026-09-05 19:27:21',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='The current time is **7:27:21 PM** (September 5, 2026) and the weather is a **beautiful day**!', type='text')]}]